In [13]:
%pip install pandas numpy matplotlib yfinance fredapi pandas-datareader pathlib

Note: you may need to restart the kernel to use updated packages.


In [15]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf
from fredapi import Fred
import pandas_datareader as web
from pathlib import Path

print("All libraries imported successfully!")

All libraries imported successfully!


In [19]:
# Define FRED series
fred_series = [
    "FEDFUNDS",
    "CPIAUCSL",
    "UNRATE",
    "DGS10",
    "DGS2",
    "VIXCLS"
]

# Download raw FRED data
fred_raw = web.DataReader(
    fred_series,
    "fred",
    start="2005-01-01"
)

# Create data directories
data_folder = Path("data")
raw_folder = data_folder / "raw"

raw_folder.mkdir(parents=True, exist_ok=True)

# Save raw FRED data
fred_raw.to_csv(raw_folder / "fred_raw.csv")

print("FRED raw data downloaded successfully!")

FRED raw data downloaded successfully!


In [20]:
# Define Yahoo Finance tickers
tickers = [
    "SPY",
    "QQQ",
    "IWM",
    "AAPL",
    "NVDA",
    "TLT",
    "GLD"
]

# Download raw daily data
yahoo_raw = yf.download(
    tickers,
    start="2005-01-01",
    auto_adjust=True
)

# Save raw Yahoo Finance data
yahoo_raw.to_csv(raw_folder / "yahoo_raw.csv")

print("Yahoo Finance raw data downloaded successfully!")

[*********************100%***********************]  7 of 7 completed


Yahoo Finance raw data downloaded successfully!


In [23]:
#Check Data types
print("FRED DATA TYPES:")
print(fred_raw.dtypes)

print("\nYAHOO FINANCE DATA TYPES:")
print(yahoo_raw.dtypes)

FRED DATA TYPES:
FEDFUNDS    float64
CPIAUCSL    float64
UNRATE      float64
DGS10       float64
DGS2        float64
VIXCLS      float64
dtype: object

YAHOO FINANCE DATA TYPES:
Price   Ticker
Close   AAPL      float64
        GLD       float64
        IWM       float64
        NVDA      float64
        QQQ       float64
        SPY       float64
        TLT       float64
High    AAPL      float64
        GLD       float64
        IWM       float64
        NVDA      float64
        QQQ       float64
        SPY       float64
        TLT       float64
Low     AAPL      float64
        GLD       float64
        IWM       float64
        NVDA      float64
        QQQ       float64
        SPY       float64
        TLT       float64
Open    AAPL      float64
        GLD       float64
        IWM       float64
        NVDA      float64
        QQQ       float64
        SPY       float64
        TLT       float64
Volume  AAPL        int64
        GLD         int64
        IWM         int64
 

In [22]:
#Missing values
print("FRED MISSING VALUES:")
print(fred_raw.isnull().sum())

print("\nYAHOO FINANCE MISSING VALUES:")
print(yahoo_raw.isnull().sum())

FRED MISSING VALUES:
FEDFUNDS    5517
CPIAUCSL    5518
UNRATE      5518
DGS10        309
DGS2         309
VIXCLS       245
dtype: int64

YAHOO FINANCE MISSING VALUES:
Price   Ticker
Close   AAPL      0
        GLD       0
        IWM       0
        NVDA      0
        QQQ       0
        SPY       0
        TLT       0
High    AAPL      0
        GLD       0
        IWM       0
        NVDA      0
        QQQ       0
        SPY       0
        TLT       0
Low     AAPL      0
        GLD       0
        IWM       0
        NVDA      0
        QQQ       0
        SPY       0
        TLT       0
Open    AAPL      0
        GLD       0
        IWM       0
        NVDA      0
        QQQ       0
        SPY       0
        TLT       0
Volume  AAPL      0
        GLD       0
        IWM       0
        NVDA      0
        QQQ       0
        SPY       0
        TLT       0
dtype: int64


In [24]:
#Duplicate dates and duplicate rows
print("FRED DUPLICATE DATES:")
print(fred_raw.index.duplicated().sum())

print("\nYAHOO DUPLICATE DATES:")
print(yahoo_raw.index.duplicated().sum())

print("\nFRED DUPLICATE ROWS:")
print(fred_raw.duplicated().sum())

print("\nYAHOO DUPLICATE ROWS:")
print(yahoo_raw.duplicated().sum())

FRED DUPLICATE DATES:
0

YAHOO DUPLICATE DATES:
0

FRED DUPLICATE ROWS:
148

YAHOO DUPLICATE ROWS:
0


In [25]:
#Outliers
def detect_outliers(df):
    numeric_data = df.select_dtypes(include=np.number)

    Q1 = numeric_data.quantile(0.25)
    Q3 = numeric_data.quantile(0.75)

    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = (
        (numeric_data < lower_bound) |
        (numeric_data > upper_bound)
    )

    return outliers.sum()


print("FRED OUTLIERS:")
print(detect_outliers(fred_raw))

print("\nYAHOO FINANCE OUTLIERS:")
print(detect_outliers(yahoo_raw))

FRED OUTLIERS:
FEDFUNDS      0
CPIAUCSL      0
UNRATE        2
DGS10         0
DGS2          0
VIXCLS      265
dtype: int64

YAHOO FINANCE OUTLIERS:
Price   Ticker
Close   AAPL       80
        GLD       449
        IWM         0
        NVDA      872
        QQQ       142
        SPY        61
        TLT         0
High    AAPL       76
        GLD       449
        IWM         0
        NVDA      872
        QQQ       142
        SPY        52
        TLT         1
Low     AAPL       83
        GLD       449
        IWM         0
        NVDA      872
        QQQ       144
        SPY        64
        TLT         0
Open    AAPL       77
        GLD       449
        IWM         0
        NVDA      872
        QQQ       143
        SPY        62
        TLT         1
Volume  AAPL      243
        GLD       293
        IWM       303
        NVDA      198
        QQQ       306
        SPY       351
        TLT       503
dtype: int64


In [26]:
# Compare available dates
fred_dates = set(fred_raw.index)
yahoo_dates = set(yahoo_raw.index)

common_dates = fred_dates.intersection(yahoo_dates)

fred_only = fred_dates - yahoo_dates
yahoo_only = yahoo_dates - fred_dates

print("FRED observations:", len(fred_dates))
print("Yahoo observations:", len(yahoo_dates))
print("Common dates:", len(common_dates))
print("Dates only in FRED:", len(fred_only))
print("Dates only in Yahoo:", len(yahoo_only))

FRED observations: 5779
Yahoo observations: 5502
Common dates: 5501
Dates only in FRED: 278
Dates only in Yahoo: 1


In [33]:
# Convert FRED data to monthly frequency
fred_monthly = fred_raw.resample("ME").last()

# Extract adjusted closing prices from Yahoo Finance
yahoo_close = yahoo_raw["Close"]

# Convert Yahoo Finance data to monthly frequency
yahoo_monthly = yahoo_close.resample("ME").last()

# Remove the last line because of insufficient data for the current month
fred_monthly = fred_monthly.iloc[:-1]
yahoo_monthly = yahoo_monthly.iloc[:-1]

In [34]:
# Synchronize FRED and Yahoo Finance data
combined_monthly = pd.concat(
    [fred_monthly, yahoo_monthly],
    axis=1,
    join="inner"
)

# Sort observations chronologically
combined_monthly = combined_monthly.sort_index()

In [39]:
# Check final dataset
print("Final dataset information:")
combined_monthly.info()

print("\nMissing values:")
print(combined_monthly.isnull().sum())

print("\nDate range:")
print("Start:", combined_monthly.index.min())
print("End:", combined_monthly.index.max())

print("\nNumber of observations:", len(combined_monthly))

# Save synchronized dataset
combined_monthly.to_csv(
    data_folder / "combined_monthly.csv"
)

print("\nCombined dataset saved successfully!")

Final dataset information:
<class 'pandas.DataFrame'>
DatetimeIndex: 262 entries, 2005-01-31 to 2026-10-31
Freq: ME
Data columns (total 13 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   FEDFUNDS  262 non-null    float64
 1   CPIAUCSL  261 non-null    float64
 2   UNRATE    261 non-null    float64
 3   DGS10     262 non-null    float64
 4   DGS2      262 non-null    float64
 5   VIXCLS    262 non-null    float64
 6   AAPL      262 non-null    float64
 7   GLD       262 non-null    float64
 8   IWM       262 non-null    float64
 9   NVDA      262 non-null    float64
 10  QQQ       262 non-null    float64
 11  SPY       262 non-null    float64
 12  TLT       262 non-null    float64
dtypes: float64(13)
memory usage: 28.7 KB

Missing values:
FEDFUNDS    0
CPIAUCSL    1
UNRATE      1
DGS10       0
DGS2        0
VIXCLS      0
AAPL        0
GLD         0
IWM         0
NVDA        0
QQQ         0
SPY         0
TLT         0
dtype: int64

Date range:

In [40]:
# Show the one date with the missing data
print(combined_monthly[combined_monthly.isnull().any(axis=1)])

            FEDFUNDS  CPIAUCSL  UNRATE  DGS10  DGS2  VIXCLS       AAPL  \
2025-10-31      4.09       NaN     NaN   4.11   3.6   17.44  269.17041   

                   GLD         IWM        NVDA         QQQ         SPY  \
2025-10-31  368.119995  243.753464  202.006027  626.128174  674.780029   

                  TLT  
2025-10-31  85.925858  
